# SCS 3546: Deep Learning
> **Assignment 3: Contextualized Word Embeddings**

### Your name & student number:

<pre> Cameron Turner </pre>

<pre> X606793 </pre>

## **Assignment Description**
***

Search Engines are a standard tool for finding relevant content. The calculation of similarity between textual information is an important factor for better search results.

### **Objectives**

**Your goal in this assignment is to calculate the textual similarity between queries and the provided sample documents, using a variety of NLP approaches.**

In achieving the above goal, you will also:
- Demonstrate how to preprocess text and embed textual data.
- Compare the results of textual similarity scoring between traditional and deep-learning based NLP methods.

### **Data and Queries**

You will use the document repository provided by `sample_repository.json`, which you can download from the following link, or from the assignment description in Quercus: https://q.utoronto.ca/courses/286389/files/21993451/download?download_frd=1

The queries you will run against these sample documents are the following:

- Query 1: “fruits”
- Query 2: “vegetables”
- Query 3: “healthy foods in Canada”

### **Techniques to Demonstrate**

The techniques you will use to compute the similarity scores are:
- 1. TF-IDF.
- 2. Semantic similarity using GloVe word vectors.
- 3. Semantic similarity using a BERT-based model.


### **Feel Free to Choose Your Own Approach**

How you go about demonstrating each of the above techniques is up to you. You are not expected to use any particular library. The code below is just meant to provide you with some guidance to get started. You **do**, however, need to demonstrate obtaining similarity scores **with all 3 techniques above**, but how you go about doing this is totally up to you. The evaluation will be based on your ability obtain results using all three techniques, plus your discussion/comparison of any differences you observe.



## **Grade Allocation**
***
15 points total

- Experiment 1 (TD-IDF), implementation: 2 marks
- Experiment 2 (GloVe), implementation: 3 marks
- Experiment 3 (BERT), implementation: 3 marks
- Comparison and Discussion: 3 marks
  - Compare all three techniques and interpret your findings. Do your best to explain the differences you observe in terms of concepts learned in class (not just the _what_, but also the _how_ and _why_ one technique produces different results from another).
- Text Pre-Processing: 2 marks
 - Cleaning and standardization (e.g. lemmatization, stemming) in Experiment 1
 - Basic text cleaning (e.g. removal of special characters or tags) in Experiments 2 and 3.
- Clarity: 2 marks
 - The marks for clarity are awarded for code documentation, clean code (e.g. avoiding repetition by building re-usable functions)  and how well you explained/supported your answers, including the use of visualizations.


# Setup and Data Import
***
You can use the code snippets below to help you load and extract the document repository.


In [1]:
# Turn off warnings.

import warnings
warnings.filterwarnings("ignore")


In [2]:
# mount the colab
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import json

# Load the sample repository JSON file.

with open('/content/drive/MyDrive/sample_repository.json') as in_file:
    repo_data = json.load(in_file)

# Extract the list of titles and documents from the data.

titles = [item[0] for item in repo_data['data']]
documents = [item[1] for item in repo_data['data']]


In [4]:
# See how many documents have been loaded.

print ("The number of documents loaded is", len(documents))

The number of documents loaded is 32


In [5]:
# let's take a look at some of these documents and titles;
# here we print the five last entries
for id in range(-5, 0, 1):
  print(f"Document title: {titles[id]}")
  print(f"Document contents: {documents[id]}")
  print("\n") # adds newline

Document title: botany
Document contents: Botany, also called plant science(s), plant biology or phytology, is the science of plant life and a branch of biology. A botanist, plant scientist or phytologist is a scientist who specialises in this field. 


Document title: Ford Bronco 
Document contents: The Ford Bronco is a model line of sport utility vehicles manufactured and marketed by Ford. ... The first SUV model developed by the company, five generations of the Bronco were sold from the 1966 to 1996 model years. A sixth generation of the model line is sold from the 2021 model year. the Ford Bronco will be available in Canada, with first deliveries beginning in spring of 2021. The Bronco will come in six versions in Canada: Base, Big Bend, Black Diamond, Outer Banks, Wildtrak and Badlands. 


Document title: List of fruit dishes
Document contents: Fruit dishes are those that use fruit as a primary ingredient. Condiments prepared with fruit as a primary ingredient are also included in

### Basic Search

Start with a basic search to get a general idea of which documents have the significant search terms.


In [6]:
# Define the query terms.

query_terms = ['fruits', 'vegetables', 'healthy', 'foods', 'Canada']

# Iterate over the query terms.

for query_term in query_terms:

    print ("Evaluating", query_term)
    print ("-----------------------")

    # Iterate over the documents.

    for id in range(0, len(documents), 1):

        # Determine if the query term is in the document.

        if (query_term in documents[id]):

            # It is, print out the title and the document.

            print("*** Found search query", query_term, "in document", id, "!!!")

            print(f"Document title: {titles[id]}")
            print(f"Document contents: {documents[id]}")

            print("\n")

Evaluating fruits
-----------------------
*** Found search query fruits in document 6 !!!
Document title: Food classes
Document contents: To a botanist, a fruit is an entity that develops from the fertilized ovary of a flower. This means that tomatoes, squash, pumpkins, cucumbers, peppers, eggplants, corn kernels, and bean and pea pods are all fruits; so are apples, pears, peaches, apricots, melons and mangos


*** Found search query fruits in document 10 !!!
Document title: Canada's Food Guide
Document contents: Canada's Food Guide is a nutrition guide produced by Health Canada to promote Healthy behaviours and habits, and lifestyles in Canada - this is to increase the number of healthy people in Canada. In 2007, it was reported to be the second most requested Canadian government publication, behind the Income Tax Forms. The Health Canada website states: Food guides are basic education tools that are designed to help people follow a healthy diet. The Guide recommends eating a variety 

### Basic Search Summary

The query term, _fruits_, was found in two documents.

The query term, _vegetables_, was found in one document.

The query term, _healthy foods in Canada_, was found in seven documents when each word is used as its own query term.

# Experiment 1: TF-IDF
***

**T**erm **F**requency - **I**nverse **D**ocument **F**requency (TF-IDF) is a traditional NLP technique to look at words that appear in both pieces of text, and score them based on how often they appear. For this experiment, you are free to use the TF-IDF implementation provided by scikit-learn.


In [7]:
# Import libraries required for TF-IDF implementation.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np
import re

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

nltk.download('punkt')
stop_words = set(stopwords.words('english'))

# Lemmatization
# e.g. you can use a lemmatizer to reduce words down to their
# simplest 'lemma' (helpful when dealing with plurals)

from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [8]:
# Let's take a look at the stop words that will be removed from the vocabulary to reduce noise.

print ("Stop words:", stop_words)

Stop words: {'ve', 'nor', 'your', 'such', 'under', 'which', 'from', "that'll", 'up', "wasn't", 'being', 'doesn', 'ma', "mustn't", "she'd", "we're", "weren't", 'theirs', 'mightn', 'other', 'didn', 'after', "don't", 'further', 'when', 'with', 'through', 'both', 'about', 'while', 'she', 'he', 'than', 'once', 'they', 'isn', 'shan', "we'll", 'as', 'against', 'but', 'd', "mightn't", "it'll", "shouldn't", 'by', 'to', 'for', 'same', 'will', 'or', 'all', 'am', "we've", 'on', 're', 'doing', 'i', 'wouldn', 'are', 'out', 'should', "haven't", 'at', 'between', "you're", 'wasn', 'yours', 'couldn', 'few', 'too', 'now', 'there', 'was', 'yourselves', 'off', 'be', 't', 'won', 'him', "you'd", 'my', 'himself', 's', 'until', "aren't", "shan't", 'over', 'and', 'myself', 'our', 'mustn', 'hadn', 'who', 'haven', 'weren', "you've", "hadn't", 'some', 'y', 'most', 'ours', 'so', "we'd", "it'd", "should've", 'if', 'above', 'does', 'into', "she'll", "hasn't", 'll', "i'll", 'shouldn', "they're", 'of', "it's", "needn't

In [9]:
# print_vocabulary

def print_vocabulary(vectorizer):

    # Print the vocabulary of all documents.

    print ("The vocabulary of all documents is as follows:")
    print (vectorizer.get_feature_names_out())


# print_vocabulary_rating

def print_vocabulary_rating(document_index, vectorizer, vectors):

    feature_names = vectorizer.get_feature_names_out()

    # Iterate over the feature names and print the word and its
    # score.

    for word, score in zip(feature_names, vectors.toarray()[document_index]):
        print(word, ":", score)


# print_tf_idf_for_document

def print_tf_idf_for_document(vectors, document_id):

    # Print the TF-IDF matrix of the document to show the term frequency and
    # Inverse Document Frequency values.

    # TF-IDF = TF × IDF

    # A high TF-IDF score means that the word is distinct as it
    # appears frequently in one document but not others.

    # A low TF-IDF score typically means that the word is more common
    # and less distinct to the document.

    print (vectors.toarray()[document_id])


# lemmatize_text

def lemmatize_text(text):

    # Create the lemmatizer object.

    lemmatizer = WordNetLemmatizer()

    # Lemmatize the tokens in the text.

    tokens = nltk.word_tokenize(text.lower())
    return [lemmatizer.lemmatize(t) for t in tokens]


# remove_punctuation

def remove_punctuation(text):

    # Invoke a regular expression to remove punctuation
    # and only contain word characters (A..Z, 0..9 etc.)
    # and whitepace.

    text = re.sub(r'[^\w\s]', '', text)
    return text


# basic_vectorizer

def basic_vectorizer(query, documents):

    # Create a basic TF-IDF vectorizer using only the
    # stop words.

    vectorizer = TfidfVectorizer(stop_words=list(stop_words))

    # Fit the query term and documents into the TF-IDF vectorizer model.

    vectors = vectorizer.fit_transform([query] + documents)

    return vectorizer, vectors


# cleaning_vectorizer

def cleaning_vectorizer(query, documents):

    # Create a cleaned TF-IDF vectorizer using the stop words,
    # lemmatized text and removal of punctuation.

    vectorizer = TfidfVectorizer(stop_words=list(stop_words), tokenizer=lemmatize_text, preprocessor=remove_punctuation)

    # Fit the query term and documents into the TF-IDF vectorizer model.

    vectors = vectorizer.fit_transform([query] + documents)

    return vectorizer, vectors


# display_similarities

def display_similarities(query, vectors):

    # Calculate the similarities matrix for the 32 documents.
    # From the fit_transform method, he query is in vectors[0:1]
    # and the documents are in vectors[1:].

    similarities = cosine_similarity(vectors[0:1], vectors[1:])

    # Print the similarities.

    print("Query:", query)
    print("--------------\n")
    print("Query similarities matrix for the documents:\n\n", similarities[0])

    indices = np.argsort(similarities[0])[::-1]

    print("\n\nDocument Indices Sorted by Similarity:")
    print("--------------------------------------\n\n")
    print(indices)

    # Print out the top 5.

    # Note that for "healthy foods in Canada" it will match documents with just 'Canada'

    print("\n\nTop 5 Documents by Similarity:")
    print("------------------------------\n\n")

    for id in indices[0:5]:
        print("Document", id, "(", similarities[0][id], ")")
        print(documents[id], "\n\n")


# print_vocabulary

def print_vocabulary(vectorizer):

    # Print the vocabulary.

    print ("The vocabulary of all documents is as follows:")
    print (vectorizer.get_feature_names_out())




In [10]:
# Set the query terms.

query_terms = [ 'fruits', 'vegetables', 'healthy foods in Canada']

# Iterate over the query terms.

for query_term in query_terms:

    # Create the TF-IDF vector based on the query term and documents.

    vectorizer, vectors = basic_vectorizer(query_term, documents)

    # Display the similarities of the query term to the documents.

    display_similarities(query_term, vectors)

    print("\n\n")


Query: fruits
--------------

Query similarities matrix for the documents:

 [0.         0.         0.         0.         0.         0.
 0.16456282 0.         0.         0.         0.07331002 0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.        ]


Document Indices Sorted by Similarity:
--------------------------------------


[ 6 10 31 30 27 26 29 28 23 22 21 20 19 18 25 24 16 17 13 15 12 11  9 14
  8  7  5  4  3  2  1  0]


Top 5 Documents by Similarity:
------------------------------


Document 6 ( 0.1645628239121347 )
To a botanist, a fruit is an entity that develops from the fertilized ovary of a flower. This means that tomatoes, squash, pumpkins, cucumbers, peppers, eggplants, corn kernels, and bean and pea pods are all fruits; so are apples, pears, peaches, apricots, melons and mangos 


Document 10 ( 0.07331001792790331 )
Canada's 

### TF-IDF Basic Summary

In the basic TF-IDF search, only a strict term match counts.  Therefore, we are looking for exactly _fruits_ or _vegetables_ or in the case of _healthy foods in Canada_, either of the non-stop-word words, _healthy_, _foods_, _Canada_.

The search term, _fruits_, matched on two documents with a highest TF-IDF rating of **.164** which is weakly relevant.

The search term, _vegetables_, matched on one document with a TF-IDF rating of **.080** which is weakly relevant.

The search term, _healthy foods in Canada_, matched on five documents with a highest TF-IDF rating of **.598** which is fairly relevant.

Also of note is that for the single search terms such as _fruits_ and _vegetables_, the TF-IDF query returns the same matches as the original _Basic Search_.

## Repeat the same task after some preprocessing

Use a minimum of 2 different text cleaning/standardization techniques (e.g. lemmatization, removing punctuation, etc).

Testing lemmatization (returning a word to its base form by removing plurals, present participle etc.) and cleaning documents of punctuation using the __cleaningvectorizer_ function.

In [11]:
# Iterate over the same set of query terms.

for query_term in query_terms:

    # Create the lemmatized, cleaning (remove punctuation) TF-IDF vector
    # based on the query term and documents.

    vectorizer, vectors = cleaning_vectorizer(query_term, documents)

    # Display the similarities of the query term to the documents.

    display_similarities(query_term, vectors)

    print("\n\n")


Query: fruits
--------------

Query similarities matrix for the documents:

 [0.15009964 0.05141357 0.         0.         0.         0.
 0.2541761  0.         0.         0.         0.0540079  0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.45421966
 0.         0.09234219]


Document Indices Sorted by Similarity:
--------------------------------------


[29  6  0 31 10  1 30 28 23 22 21 20 27 26 25 24 16 17 18 19 12 15 13 14
  8  9 11  7  4  5  2  3]


Top 5 Documents by Similarity:
------------------------------


Document 29 ( 0.4542196580921685 )
Fruit dishes are those that use fruit as a primary ingredient. Condiments prepared with fruit as a primary ingredient are also included in this list. 


Document 6 ( 0.25417610260016693 )
To a botanist, a fruit is an entity that develops from the fertilized ovary of a flower. This means that tomatoes, squash, pum

### TF-IDF Lemmatization Summary

In the lemmatized / cleaned TF-IDF search, there is more flexibility in terms of relevant matches.  For example, plurals are counted the same as the word's singular form and punctuation is removed such that _fruits_ provides the same relevance as _fruits;_.

The search term, _fruits_, matched on six documents with a highest TF-IDF rating of **.454** which is fairly relevant.

The search term, _vegetables_, still only matched on one document with a similar TF-IDF rating of **.076** which is weakly relevant.

The search term, _healthy foods in Canada_, matched on six documents with a highest TF-IDF rating of **.625** which is pretty good relevance.

### TF-IDF Experiment Summary

The observed results are what would be expected after lemmatization.  By discarding differences based on plularity and punctuation, it would be expected that the query term would result in more relevant document matches as it is not constrained by exact text matches.

Except for the _vegetables_ query which had a similiar result (one match and a TF-IDF rating of ~0.08), the search results for the other results yielded more document matches (four more for _fruits_ and one more for _healthy foods in Canada_) and there were stronger relevance matches as well (.454 for _fruits_ compared to .164; and .625 for _healthy foods in Canada_ compared to .598).

In [12]:
# your response here

# Experiment 2: Semantic matching using GloVe embeddings
***

In [13]:
# Ensure gensim package is installed.

!pip install --upgrade pip
!pip install gensim

In [14]:
import sys

# if you decide to use the gensim library and the sample codes below,
# you would need gensim version >=4.0.1 to be installed
# The current Python version (3.12.13) seems to have compatibility issues with gensim==4.0.1.
# Attempting to install the latest compatible version of gensim instead.

# First, uninstall any potentially broken gensim installation
#!pip uninstall -y gensim

# Then, install gensim without a specific version to get the latest compatible one
#!pip install gensim

import gensim
print(f"Gensim version: {gensim.__version__}")

Gensim version: 4.4.0


In [15]:
# Import libraries required for gensim.

import logging
import json
import logging
from re import sub
from multiprocessing import cpu_count

import numpy as np

import gensim.downloader as api
from gensim.utils import simple_preprocess
from gensim.corpora import Dictionary
from gensim.models import TfidfModel
from gensim.similarities import WordEmbeddingSimilarityIndex
from gensim.similarities import SparseTermSimilarityMatrix
from gensim.similarities import SoftCosineSimilarity

In [16]:
# optional, but it helps
import logging

# Initialize logging.
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.WARNING)

In [17]:
import nltk

# Import and download stopwords from NLTK.
nltk.download('stopwords')  # Download stopwords list.
stopwords = set(nltk.corpus.stopwords.words("english"))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [18]:
def preprocess(doc):
    # Tokenize, clean up input document string
    doc = sub(r'<img[^<>]+(>|$)', " image_token ", doc)
    # you may decide to add additional steps here
    return [token for token in simple_preprocess(doc, min_len=0, max_len=float("inf")) if token not in stopwords]

In [19]:
# Download and load the GloVe word vector embeddings
if 'glove' not in locals():  # only load if not already in memory
    glove = api.load("glove-wiki-gigaword-50")

similarity_index = WordEmbeddingSimilarityIndex(glove)

In [20]:
# setup_GLove

def setup_GLove(query_term):

    # Preprocess the documents, including the query string

    corpus = [preprocess(document) for document in documents]
    query = preprocess(query_term)

    # Build the term dictionary, TF-idf model
    # Keep in mind that the search query must be in the dictionary as well, in case the terms do not overlap with the documents

    dictionary = Dictionary(corpus+[query])
    tfidf = TfidfModel(dictionary=dictionary)

    # Create the term similarity matrix.
    # The nonzero_limit enforces sparsity by limiting the number of non-zero terms in each column.
    # In my case, I got best results by removing the default value of 100

    similarity_matrix = SparseTermSimilarityMatrix(similarity_index, dictionary, tfidf)  # , nonzero_limit=None)

    # Compute similarity measure between the query and the documents.

    query_tf = tfidf[dictionary.doc2bow(query)]

    index = SoftCosineSimilarity(
                tfidf[[dictionary.doc2bow(document) for document in corpus]],
                similarity_matrix)

    doc_similarity_scores = index[query_tf]

    return doc_similarity_scores


# display_GLoVe_similarities

def display_GLoVe_similarities(query, similarities):

    # Print the similarities.

    print("Query:", query)
    print("--------------\n")
    print("Query similarities matrix for the documents:\n\n", similarities)  #[0])

    indices = np.argsort(similarities)[::-1]

    print("\n\nDocument Indices Sorted by Similarity:")
    print("--------------------------------------\n\n")
    print(indices)

    # Print out the top 5.

    print("\n\nTop 5 Documents by Similarity:")
    print("------------------------------\n\n")

    for id in indices[0:5]:
        print("Document", id, "(", similarities[id], ")")
        print(documents[id], "\n\n")



In [21]:
# Calculate the similarity scores for the fruits query term with the documents.

similarity_scores = setup_GLove("fruits")

# Display the similarities.

display_GLoVe_similarities("fruits", similarity_scores)

100%|██████████| 568/568 [00:08<00:00, 68.91it/s]

Query: fruits
--------------

Query similarities matrix for the documents:

 [0.8091958  0.72479224 0.         0.         0.41293627 0.
 0.88390946 0.         0.5453234  0.5064042  0.7613044  0.
 0.         0.5223634  0.         0.5223635  0.56572443 0.41293627
 0.45817053 0.         0.         0.50264883 0.50177664 0.63024676
 0.         0.69269216 0.         0.         0.         0.8436877
 0.         0.8436877 ]


Document Indices Sorted by Similarity:
--------------------------------------


[ 6 31 29  0 10  1 25 23 16  8 15 13  9 21 22 18  4 17 26 20 28 27 24 30
 19 11 14 12  5  7  2  3]


Top 5 Documents by Similarity:
------------------------------


Document 6 ( 0.88390946 )
To a botanist, a fruit is an entity that develops from the fertilized ovary of a flower. This means that tomatoes, squash, pumpkins, cucumbers, peppers, eggplants, corn kernels, and bean and pea pods are all fruits; so are apples, pears, peaches, apricots, melons and mangos 


Document 31 ( 0.8436877 )
A fr

In [22]:
# Calculate the similarity scores for the vegetables query term with the documents.

similarity_scores = setup_GLove("vegetables")

# Display the similarities.

display_GLoVe_similarities("vegetables", similarity_scores)

100%|██████████| 568/568 [00:06<00:00, 81.44it/s]

Query: vegetables
--------------

Query similarities matrix for the documents:

 [0.7264673  0.6681484  0.         0.         0.70769125 0.
 0.8961465  0.         0.4080156  0.6793368  0.8103701  0.
 0.         0.65738994 0.30672875 0.65103805 0.41913503 0.7504946
 0.62305063 0.         0.         0.40439582 0.6066437  0.48425215
 0.         0.5168466  0.         0.         0.         0.760345
 0.         0.65949464]


Document Indices Sorted by Similarity:
--------------------------------------


[ 6 10 29 17  0  4  9  1 31 13 15 18 22 25 23 16  8 21 14 26 28 27 24 30
 19 20 11 12  5  7  2  3]


Top 5 Documents by Similarity:
------------------------------


Document 6 ( 0.8961465 )
To a botanist, a fruit is an entity that develops from the fertilized ovary of a flower. This means that tomatoes, squash, pumpkins, cucumbers, peppers, eggplants, corn kernels, and bean and pea pods are all fruits; so are apples, pears, peaches, apricots, melons and mangos 


Document 10 ( 0.8103701 )
Can

In [23]:
# Calculate the similarity scores for the "healthy foods in Canada" query term with the documents.

similarity_scores = setup_GLove("healthy foods in Canada")

# Display the similarities.

display_GLoVe_similarities("healthy foods in Canada", similarity_scores)

100%|██████████| 568/568 [00:08<00:00, 69.88it/s]

Query: healthy foods in Canada
--------------

Query similarities matrix for the documents:

 [0.5269453  0.4567337  0.5259751  0.5259751  0.43356127 0.5259752
 0.338879   0.44529146 0.3543496  0.68354654 0.93875146 0.34576914
 0.39975703 0.32097164 0.23889096 0.32097158 0.58788824 0.30108258
 0.32854313 0.5259752  0.5092128  0.         0.34958506 0.
 0.         0.31835467 0.17225574 0.17917013 0.4114949  0.3850944
 0.3172169  0.5886519 ]


Document Indices Sorted by Similarity:
--------------------------------------


[10  9 31 16  0  5 19  2  3 20  1  7  4 28 12 29  8 22 11  6 18 13 15 25
 30 17 14 27 26 23 24 21]


Top 5 Documents by Similarity:
------------------------------


Document 10 ( 0.93875146 )
Canada's Food Guide is a nutrition guide produced by Health Canada to promote Healthy behaviours and habits, and lifestyles in Canada - this is to increase the number of healthy people in Canada. In 2007, it was reported to be the second most requested Canadian government publicatio

## GLoVE Experiment Summary

Once semantics are used in the search, far more documents match the query term for word similarity (i.e., not exact matches).  There are 18 - 20 documents that match for _fruits_ and _vegetables_ and 29 documents match for _healthy foods in Canada_.

The top scores indicate stronger similarity matches as well.  For _fruits_, the top document match had a similarity score of **.884**.  For _vegetables_, the top document match had a similarity score of **.896**.  For _healthy foods in Canada_, the top document match had a similarity score of **.939**.  All of these are strong relevance scores for matching words based on similarity.

Therefore, adding a semantic search resulted in more documents that matched with higher scores as word similarity was being measured instead of exact string matches.

# Experiment 3: BERT Model
***
Use a BERT model obtain sentence embeddings and calculate the similarity between queries and documents.

> Hint: see the Module 07 jupyter notebook for examples of how to work with BERT.

In [24]:
#!pip install sentence-transformers

In [25]:
# Import libraries required for sentence BERT.

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# bert_query

def bert_query(query_term):

    # Create the BERT model.

    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Encode the documents and query.

    documents_embeddings = model.encode(documents)
    query_embedding = model.encode([query_term])

    # Calculate the similarities.

    similarities = cosine_similarity(query_embedding, documents_embeddings)[0]

    # Display the similarity scores.

    print(similarities)

    # Sort the similarity indices.

    indices = np.argsort(similarities)[::-1]

    print("\n\nDocument Indices Sorted by Similarity:")
    print("--------------------------------------\n\n")
    print(indices)

    # Print out the top 5.

    print("\n\nTop 5 Documents by Similarity:")
    print("------------------------------\n\n")

    for id in indices[0:5]:
          print("Document", id, "(", similarities[id], ")")
          print(documents[id], "\n\n")



In [26]:
bert_query("fruits")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[ 0.36305392  0.3082934   0.2301707   0.2301707   0.23809604  0.2259947
  0.6040926   0.1621491   0.19516818  0.26960295  0.16290572  0.05737146
  0.03478238  0.24359155  0.3469388   0.23277631  0.28438804  0.24617276
  0.4570652   0.2259947   0.20843676  0.36520165  0.39910865  0.40421194
  0.01113953  0.1116775   0.04077464  0.23539121  0.02426565  0.5436354
 -0.04768821  0.436352  ]


Document Indices Sorted by Similarity:
--------------------------------------


[ 6 29 18 31 23 22 21  0 14  1 16  9 17 13  4 27 15  2  3  5 19 20  8 10
  7 25 11 26 12 28 24 30]


Top 5 Documents by Similarity:
------------------------------


Document 6 ( 0.6040926 )
To a botanist, a fruit is an entity that develops from the fertilized ovary of a flower. This means that tomatoes, squash, pumpkins, cucumbers, peppers, eggplants, corn kernels, and bean and pea pods are all fruits; so are apples, pears, peaches, apricots, melons and mangos 


Document 29 ( 0.5436354 )
Fruit dishes are those that use fru

In [27]:
bert_query("vegetables")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[ 0.20347494  0.16656762  0.1493096   0.1493096   0.3536978   0.14375311
  0.478733    0.20808162  0.3548925   0.31390396  0.28699046  0.10639323
  0.05539191  0.34616035  0.27739727  0.3512717   0.26159608  0.38259315
  0.3914768   0.14375311  0.11903963  0.22941718  0.24459636  0.24073723
  0.02735085  0.20284955 -0.05048694  0.25157228  0.06823417  0.42798805
 -0.06258343  0.32930547]


Document Indices Sorted by Similarity:
--------------------------------------


[ 6 29 18 17  8  4 15 13 31  9 10 14 16 27 22 23 21  7  0 25  1  2  3 19
  5 20 11 28 12 24 26 30]


Top 5 Documents by Similarity:
------------------------------


Document 6 ( 0.478733 )
To a botanist, a fruit is an entity that develops from the fertilized ovary of a flower. This means that tomatoes, squash, pumpkins, cucumbers, peppers, eggplants, corn kernels, and bean and pea pods are all fruits; so are apples, pears, peaches, apricots, melons and mangos 


Document 29 ( 0.42798805 )
Fruit dishes are those that use f

In [28]:
bert_query("healthy foods in Canada")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[ 0.1304245   0.16457224  0.09981206  0.09981206  0.13976115  0.07659147
  0.17359266  0.16948284  0.24629483  0.23942584  0.6387734   0.06662554
  0.38549566  0.10586663  0.20125441  0.09894903  0.201063    0.23805095
  0.34801754  0.07659147  0.04205973  0.11682644  0.16000822  0.08817965
  0.03449418  0.0251135  -0.06960224  0.01924041  0.13642916  0.3016522
 -0.07274953  0.21513559]


Document Indices Sorted by Similarity:
--------------------------------------


[10 12 18 29  8  9 17 31 14 16  6  7  1 22  4 28  0 21 13  2  3 15 23 19
  5 11 20 24 25 27 26 30]


Top 5 Documents by Similarity:
------------------------------


Document 10 ( 0.6387734 )
Canada's Food Guide is a nutrition guide produced by Health Canada to promote Healthy behaviours and habits, and lifestyles in Canada - this is to increase the number of healthy people in Canada. In 2007, it was reported to be the second most requested Canadian government publication, behind the Income Tax Forms. The Health Canada webs

## BERT Experiment Summary

All three query terms resulted in some form of matching with all of the documents at various score strength.

As BERT calculates similarity scores different than TF-IDF and GLoVE, the similarity scores aren't necessarily going to be higher.

The top similarity scores are generally lower than GLoVe due to GLoVe calculating scores based on the semantics of words (e.g., car = automobile) while BERT calculates similarity based on the sentence as a whole (e.g., He wants to purchase a car; They acquired a new car).

 # Technique Comparison
 ***

Compare all three techniques and interpret your findings. Do your best to explain the differences you observe in terms of concepts learned in class (not just the what, but also the how and why one technique produces different results from another).


TF-IDF calculates similarity between query terms and documents based on string matches of words.  In its simplest terms, it only matches by string comparison.  When lemmatized and cleaned, it can interpret the similarities between _fruit_ and _fruits_ or _fruit;_.

GLoVe adds semantic matching.  Therefore, it knows that apples and bananas are fruits so these terms would match for a query, _fruits_.  This allows a search to cast a wider net.

BERT matches by sentence and context.  Therefore, it looks for similar meaning, not just exact text matches.  

Therefore, as they are all different, the similarity scores can't really be compared between the three techniques.

Which one is better?

That depends on the purpose of the search.  

BERT can give you better results if you're not looking for exact search matches but rather looking for a wider context in your search.  For example, one may like to see results on car repair without necessarily specifying the vocabulary.

GLoVe can give better results if you are looking for more precise matches but don't want exact matches.  For example, one may be looking for fruits but the actual term _fruits_ does not have to be part of the document.

Sometimes, though, one may be looking for exact matches, especially for specialized keywords in contract documents, for example.  In this case, TF-IDF would be a better choice as one would only want results to be returned that are an exact, clear match.  

Therefore, each search technique has its strengths and which one to use depends on the purpose of the query search.